# NB01 — Exploration des données : séquences VHH (OAS) ou peptides (UniProt)

**Projet 1.1 — Tâches 1.1 à 1.4 du plan**

## Objectifs
- Charger et filtrer un jeu de séquences protéiques réelles.
- Produire les statistiques descriptives de base : longueurs, composition en acides aminés.
- Construire une matrice de poids positionnels (PWM) et un logo de séquences.
- Détecter les doublons exacts.

## Références
- Olsen T.H. et al. (2022). *Observed Antibody Space: A Resource for Data Mining Next-Generation Sequencing of Antibody Repertoires.* Journal of Immunology. — base OAS : https://opig.stats.ox.ac.uk/webapps/oas/
- UniProt Consortium (2023). *UniProt Knowledgebase 2023.* Nucleic Acids Research 51(D1).
- Schneider T.D., Stephens R.M. (1990). *Sequence logos: a new way to display consensus sequences.* Nucleic Acids Research 18(20).

## Données
Pour ce notebook on suppose un fichier CSV local avec une colonne `sequence` (une séquence par ligne).
Sources possibles :
- OAS (VHH / nanobodies) : téléchargement unitaire via https://opig.stats.ox.ac.uk/api/v1/sequence/ ou en masse via la page de téléchargement.
- UniProt : requête telle que https://rest.uniprot.org/uniprotkb/stream?query=(length:[10 TO 50]) — peptides courts.

## Formules utilisées (exactes, sans approximation)
- Fréquence d'un acide aminé a : f(a) = count(a) / N_total.
- Entropie de Shannon par position : H(i) = - sum_a p_i(a) log2 p_i(a) ; information du logo : I(i) = log2(20) - H(i) (Schneider & Stephens, 1990).
- Matrice PWM : PWM(i, a) = log2( p_i(a) / p_bg(a) ), où p_bg = fréquence globale observée.


In [ ]:
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
VALID_RE = re.compile(r"^[ACDEFGHIKLMNPQRSTVWY]+$")

# --- Chargement ---
# Adaptez le chemin. Colonne attendue : "sequence"
df = pd.read_csv("oas_vhh.csv")
seqs = df["sequence"].astype(str).tolist()
print("Nombre brut de séquences :", len(seqs))

# --- Filtrage exact : alphabet des 20 AA uniquement ---
seqs = [s.upper().strip() for s in seqs]
seqs = [s for s in seqs if VALID_RE.match(s)]
print("Après filtrage alphabet :", len(seqs))

# --- Statistiques de longueur ---
lengths = np.array([len(s) for s in seqs])
print("Longueur min / médiane / max :", lengths.min(), int(np.median(lengths)), lengths.max())

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(lengths, bins=60)
ax.set_xlabel("Longueur (résidus)"); ax.set_ylabel("Nombre de séquences")
ax.set_title("Distribution des longueurs")
plt.show()


In [ ]:
# --- Composition globale en acides aminés ---
counts = Counter("".join(seqs))
total = sum(counts.values())
comp = {a: counts[a] / total for a in AMINO_ACIDS}

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(list(comp.keys()), list(comp.values()))
ax.set_ylabel("Fréquence"); ax.set_title("Composition globale")
plt.show()
print({a: round(comp[a], 4) for a in AMINO_ACIDS})


In [ ]:
# --- PWM par position (sous-ensemble de longueur fixe, ex. VHH ~115-130 AA) ---
# On restreint à un intervalle de longueurs pour aligner sans gap.
L_MIN, L_MAX = 110, 130
sub = [s for s in seqs if L_MIN <= len(s) <= L_MAX]
print("Séquences dans la fenêtre :", len(sub))

# Alignement simple : troncature à la longueur minimale (analyse descriptive, PAS une véritable
# stratification de recherche — pour le VHH on utilisera l'annotation IMGT des cadres CDR en Phase 2)
L = min(len(s) for s in sub)
sub = [s[:L] for s in sub]
M = len(sub)

pwm = np.zeros((L, 20))
for i in range(L):
    for j, a in enumerate(AMINO_ACIDS):
        pwm[i, j] = np.mean([s[i] == a for s in sub])

bg = np.array([comp[a] for a in AMINO_ACIDS])
pwm_log = np.log2((pwm + 0.0) / bg[None, :])  # information relative au fond observé

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(pwm_log.T, aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3)
ax.set_yticks(range(20), list(AMINO_ACIDS))
ax.set_xlabel("Position"); ax.set_title("log2(p_i(a) / p_bg(a))")
fig.colorbar(im, ax=ax)
plt.show()


In [ ]:
# --- Entropie par position et information du logo (Schneider & Stephens 1990) ---
eps = np.finfo(float).tiny  # seulement pour éviter log2(0) : masque exact des positions sans comptage
pwm_safe = np.where(pwm > 0, pwm, 1.0)
H = -(pwm * np.log2(pwm_safe)).sum(axis=1)
I = np.log2(20) - H

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(I)
ax.set_xlabel("Position"); ax.set_ylabel("Information (bits)")
ax.set_title("Information par position")
plt.show()


In [ ]:
# --- Doublons exacts ---
unique = set(seqs)
print("Séquences uniques (identité exacte) :", len(unique), "/", len(seqs))


## Questions d'analyse (à rédiger dans le rapport)
1. La distribution des longueurs est-elle unimodale ? Y a-t-il des sous-populations ?
2. Quelles positions présentent une forte information (conservation) ? Pour le VHH, retrouve-t-on la conservation des cadres (framework) et la variabilité des CDR ?
3. Quelle fraction du jeu est redondante en identité exacte ? Pourquoi faut-il aussi dédupliquer à identité partielle (cd-hit / MMseqs2, cf. Fu et al. 2012) avant de spliter en train/test ?
